In [0]:
# ======================================
# Silver Layer Transformation Notebook
# ======================================

# Import required libraries
from pyspark.sql.functions import col, to_date, round

# -----------------------------
# 1️⃣ Read from Bronze Table
# -----------------------------
bronze_df = spark.table("jarvis_server_1.bronze.dlt_stock")
display(bronze_df)

# -----------------------------
# 2️⃣ Data Cleaning / Transformation
# -----------------------------
# Convert numeric columns to Float
silver_df = bronze_df.withColumn("open", col("open").cast("double")) \
                     .withColumn("high", col("high").cast("double")) \
                     .withColumn("low", col("low").cast("double")) \
                     .withColumn("close", col("close").cast("double")) \
                     .withColumn("volume", col("volume").cast("long"))

# Convert date column to proper date type
silver_df = silver_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

# Remove any rows with nulls in critical columns
silver_df = silver_df.dropna(subset=["symbol", "date", "open", "high", "low", "close", "volume"])

display(silver_df.limit(20))

# -----------------------------
# 3️⃣ Optional Enrichment
# -----------------------------
# Example: Calculate daily return percentage
silver_df = silver_df.withColumn("daily_return_pct", round((col("close") - col("open")) / col("open") * 100, 2))

display(silver_df.limit(20))

# -----------------------------
# 4️⃣ Write to Silver Layer
# -----------------------------
silver_df.write.format("delta") \
               .mode("overwrite") \
               .saveAsTable("jarvis_server_1.silver.stock_data")

print("Silver layer table 'jarvis_server_1.silver.stock_data' created successfully!")

close,date,high,low,open,symbol,volume
272.1400,2026-02-24,274.8900,267.7100,267.8600,AAPL,47014619
266.1800,2026-02-23,269.4300,263.3810,263.4900,AAPL,37308155
264.5800,2026-02-20,264.7500,258.1600,258.9700,AAPL,42070499
260.5800,2026-02-19,264.4800,260.0500,262.6000,AAPL,30845294
264.3500,2026-02-18,266.8200,262.4500,263.6000,AAPL,34203337
263.8800,2026-02-17,266.2900,255.5400,258.0500,AAPL,58469094
255.7800,2026-02-13,262.2300,255.4500,262.0100,AAPL,56290673
261.7300,2026-02-12,275.7200,260.1800,275.5900,AAPL,81077229
275.5000,2026-02-11,280.1800,274.4500,274.6950,AAPL,51931283
273.6800,2026-02-10,275.3700,272.9400,274.8850,AAPL,34376898


close,date,high,low,open,symbol,volume
272.14,2026-02-24,274.89,267.71,267.86,AAPL,47014619
266.18,2026-02-23,269.43,263.381,263.49,AAPL,37308155
264.58,2026-02-20,264.75,258.16,258.97,AAPL,42070499
260.58,2026-02-19,264.48,260.05,262.6,AAPL,30845294
264.35,2026-02-18,266.82,262.45,263.6,AAPL,34203337
263.88,2026-02-17,266.29,255.54,258.05,AAPL,58469094
255.78,2026-02-13,262.23,255.45,262.01,AAPL,56290673
261.73,2026-02-12,275.72,260.18,275.59,AAPL,81077229
275.5,2026-02-11,280.18,274.45,274.695,AAPL,51931283
273.68,2026-02-10,275.37,272.94,274.885,AAPL,34376898


close,date,high,low,open,symbol,volume,daily_return_pct
272.14,2026-02-24,274.89,267.71,267.86,AAPL,47014619,1.6
266.18,2026-02-23,269.43,263.381,263.49,AAPL,37308155,1.02
264.58,2026-02-20,264.75,258.16,258.97,AAPL,42070499,2.17
260.58,2026-02-19,264.48,260.05,262.6,AAPL,30845294,-0.77
264.35,2026-02-18,266.82,262.45,263.6,AAPL,34203337,0.28
263.88,2026-02-17,266.29,255.54,258.05,AAPL,58469094,2.26
255.78,2026-02-13,262.23,255.45,262.01,AAPL,56290673,-2.38
261.73,2026-02-12,275.72,260.18,275.59,AAPL,81077229,-5.03
275.5,2026-02-11,280.18,274.45,274.695,AAPL,51931283,0.29
273.68,2026-02-10,275.37,272.94,274.885,AAPL,34376898,-0.44


Silver layer table 'jarvis_server_1.silver.stock_data' created successfully!
